# EcoHome Energy Advisor - RAG Setup

In this notebook, you'll set up the Retrieval-Augmented Generation (RAG) pipeline for the EcoHome Energy Advisor. This will allow the agent to access and cite relevant energy-saving tips and best practices.

## Learning Objectives
- Set up ChromaDB vector store
- Load and process energy-saving documents
- Create embeddings for document chunks
- Implement semantic search functionality
- Test the RAG pipeline

## Documents Available
Every `.txt` file in `data/documents/`: the two starter documents (device best practices, general energy savings) and six added ones covering HVAC optimization, smart-home automation, renewable (solar) integration, seasonal energy management, energy storage, and EV charging with time-of-use pricing.


## 1. Import Required Libraries


In [1]:
# Import the necessary libraries for RAG setup
import os
from langchain_chroma  import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from dotenv import load_dotenv
from pathlib import Path


In [2]:
load_dotenv()

True

## 2. Load and Process Documents


In [3]:
# Load every energy-saving document in data/documents/
# The starter listed two files by name, so documents added later were never indexed.
# Listing the folder means the knowledge base always matches what is on disk.

documents = []
document_paths = sorted(Path("data/documents").glob("*.txt"))

for doc_path in document_paths:
    loader = TextLoader(str(doc_path), encoding="utf-8")
    docs = loader.load()
    documents.extend(docs)
    print(f"Loaded {len(docs)} document from {doc_path.as_posix()} ({len(docs[0].page_content.split())} words)")

print(f"Total documents loaded: {len(documents)}")
assert len(documents) >= 7, "expected the 2 starter documents plus at least 5 added ones"

Loaded 1 document from data/documents/tip_device_best_practices.txt (277 words)
Loaded 1 document from data/documents/tip_energy_savings.txt (223 words)
Loaded 1 document from data/documents/tip_energy_storage_optimization.txt (257 words)
Loaded 1 document from data/documents/tip_ev_charging_and_time_of_use.txt (293 words)
Loaded 1 document from data/documents/tip_hvac_optimization.txt (392 words)
Loaded 1 document from data/documents/tip_renewable_energy_integration.txt (295 words)
Loaded 1 document from data/documents/tip_seasonal_energy_management.txt (265 words)
Loaded 1 document from data/documents/tip_smart_home_automation.txt (296 words)
Total documents loaded: 8


## 3. Split Documents into Chunks


In [4]:
# Split documents into smaller chunks for better retrieval.
# load_and_split_documents (tools.py) is the same function the search tool uses, so the notebook
# and the agent always index the documents the same way. 800 characters with 120 overlap keeps
# each tip with its heading; the starter's 1000/200 often merged two unrelated sections.
from tools import load_and_split_documents, CHUNK_SIZE, CHUNK_OVERLAP

splits = load_and_split_documents()
print(f"Split {len(documents)} documents into {len(splits)} chunks (chunk_size={CHUNK_SIZE}, chunk_overlap={CHUNK_OVERLAP})")
from collections import Counter
for source, n in sorted(Counter(s.metadata['source'] for s in splits).items()):
    print(f"  {source}: {n} chunks")

# Show sample chunk
if splits:
    print(f"\nSample chunk {splits[0].id} (first 200 characters):")
    print(splits[0].page_content[:200] + "...")

Split 8 documents into 25 chunks (chunk_size=800, chunk_overlap=120)
  tip_device_best_practices.txt: 3 chunks
  tip_energy_savings.txt: 2 chunks
  tip_energy_storage_optimization.txt: 3 chunks
  tip_ev_charging_and_time_of_use.txt: 3 chunks
  tip_hvac_optimization.txt: 4 chunks
  tip_renewable_energy_integration.txt: 3 chunks
  tip_seasonal_energy_management.txt: 3 chunks
  tip_smart_home_automation.txt: 4 chunks

Sample chunk tip_device_best_practices.txt#0 (first 200 characters):
Large devices like electric vehicles, washing machines and dishwashers often support delayed start or timer functions. Schedule these devices to run outside of peak electricity pricing hours or during...


## 4. Create Vector Store


In [5]:
# Create (or rebuild) the persisted Chroma vector store.
# build_vectorstore (tools.py) embeds the chunks with stable ids, so re-running never stores duplicates,
# and records a manifest of document hashes so the search tool rebuilds the index if a document changes.
from tools import build_vectorstore, EMBEDDING_MODEL, COLLECTION_NAME, VECTORSTORE_DIR, BASE_DIR

vectorstore = build_vectorstore(force=True)

persist_directory = VECTORSTORE_DIR.relative_to(BASE_DIR).as_posix()
print(f"Vector store created and persisted to {persist_directory}")
print(f"Collection {COLLECTION_NAME!r}, embedding model {EMBEDDING_MODEL}")
print(f"Total vectors stored: {len(vectorstore.get(include=[])['ids'])}")

Vector store created and persisted to data/vectorstore
Collection 'ecohome_energy_tips', embedding model text-embedding-3-small
Total vectors stored: 25


## 5. Test the RAG Pipeline


In [6]:
# Test the search functionality
# Try different queries related to energy optimization
# Test queries like:
# - "electric vehicle charging tips"
# - "thermostat optimization"
# - "dishwasher energy saving"
# - "solar power maximization"

test_queries = [
    "electric vehicle charging tips",
    "thermostat optimization",
    "dishwasher energy saving",
    "solar power maximization",
    "HVAC system efficiency",
    "pool pump scheduling"
]

print("=== Testing Vector Search ===")
for query in test_queries:
    print(f"\nQuery: '{query}'")
    hits = vectorstore.similarity_search_with_relevance_scores(query, k=2)
    for i, (doc, score) in enumerate(hits):
        print(f"  Result {i+1} [{doc.metadata['source']}, relevance {score:.2f}]: {doc.page_content[:100]!r}...")


=== Testing Vector Search ===

Query: 'electric vehicle charging tips'


  Result 1 [tip_ev_charging_and_time_of_use.txt, relevance 0.65]: 'Charging plans\n- Overnight off-peak charging is the simplest way to cut charging costs. Set the char'...
  Result 2 [tip_ev_charging_and_time_of_use.txt, relevance 0.58]: 'EV Charging and Time-of-Use Pricing\n\nAn electric vehicle is often the largest single load in a home,'...

Query: 'thermostat optimization'


  Result 1 [tip_hvac_optimization.txt, relevance 0.58]: 'Thermostat setpoints and setbacks\n- Every degree matters. Raising the cooling setpoint or lowering t'...
  Result 2 [tip_device_best_practices.txt, relevance 0.53]: 'HVAC System Optimization:\n- Change air filters monthly for better airflow\n- Clean outdoor condenser '...

Query: 'dishwasher energy saving'


  Result 1 [tip_device_best_practices.txt, relevance 0.64]: 'Washing Machine Optimization:\n- Run full loads to maximize efficiency\n- Use cold water for most load'...
  Result 2 [tip_energy_savings.txt, relevance 0.55]: 'Saving energy at home can be simple and effective. Turn off lights when not in use and unplug device'...

Query: 'solar power maximization'


  Result 1 [tip_renewable_energy_integration.txt, relevance 0.55]: 'Renewable Energy Integration: Getting the Most from Rooftop Solar\n\nSolar panels produce the most ene'...
  Result 2 [tip_renewable_energy_integration.txt, relevance 0.54]: 'Reading solar data\n- Daily generation is shaped like a hill: low at dawn, highest near solar noon, l'...

Query: 'HVAC system efficiency'


  Result 1 [tip_hvac_optimization.txt, relevance 0.63]: "HVAC Optimization Strategies\n\nHeating and cooling are usually the largest share of a home's energy u"...
  Result 2 [tip_hvac_optimization.txt, relevance 0.59]: 'Maintenance that keeps efficiency high\n- Replace or clean air filters every one to three months; a c'...

Query: 'pool pump scheduling'


  Result 1 [tip_device_best_practices.txt, relevance 0.49]: 'HVAC System Optimization:\n- Change air filters monthly for better airflow\n- Clean outdoor condenser '...
  Result 2 [tip_smart_home_automation.txt, relevance 0.47]: 'Price-aware automation\n- Many utilities publish time-of-use schedules. A home energy app or smart hu'...


## 6. Test the Search Tool


In [7]:
# Test the search_energy_tips tool from tools.py
# Import and test the tool with various queries
# Verify that it returns relevant results

from tools import search_energy_tips

# Test the search_energy_tips function
print("=== Testing search_energy_tips Tool ===")

test_queries = [
    "electric vehicle charging",
    "thermostat settings",
    "dishwasher optimization",
    "solar power tips"
]

for query in test_queries:
    print(f"\nQuery: '{query}'")
    result = search_energy_tips.invoke(
        input={
            "query": query, 
            "max_results": 3,
        }
    )
    
    if "error" in result:
        print(f"  Error: {result['error']}")
    else:
        print(f"  Found {result['total_results']} results")
        for i, tip in enumerate(result['tips']):
            print(f"    {i+1}. {tip['content'][:100]}...")
            print(f"       Source: {tip['source']}")
            print(f"       Relevance: {tip['relevance_score']}")


=== Testing search_energy_tips Tool ===

Query: 'electric vehicle charging'


  Found 3 results
    1. EV Charging and Time-of-Use Pricing

An electric vehicle is often the largest single load in a home,...
       Source: tip_ev_charging_and_time_of_use.txt
       Relevance: 0.613
    2. Charging plans
- Overnight off-peak charging is the simplest way to cut charging costs. Set the char...
       Source: tip_ev_charging_and_time_of_use.txt
       Relevance: 0.603
    3. Large devices like electric vehicles, washing machines and dishwashers often support delayed start o...
       Source: tip_device_best_practices.txt
       Relevance: 0.532

Query: 'thermostat settings'


  Found 3 results
    1. Thermostat setpoints and setbacks
- Every degree matters. Raising the cooling setpoint or lowering t...
       Source: tip_hvac_optimization.txt
       Relevance: 0.599
    2. Winter
- Heating drives use, often in the early morning and evening. Lower the setpoint at night and...
       Source: tip_seasonal_energy_management.txt
       Relevance: 0.484
    3. Rules worth setting up
- Delay-start large appliances: dishwashers, washing machines and dryers can ...
       Source: tip_smart_home_automation.txt
       Relevance: 0.454

Query: 'dishwasher optimization'


  Found 3 results
    1. Washing Machine Optimization:
- Run full loads to maximize efficiency
- Use cold water for most load...
       Source: tip_device_best_practices.txt
       Relevance: 0.615
    2. HVAC System Optimization:
- Change air filters monthly for better airflow
- Clean outdoor condenser ...
       Source: tip_device_best_practices.txt
       Relevance: 0.464
    3. Rules worth setting up
- Delay-start large appliances: dishwashers, washing machines and dryers can ...
       Source: tip_smart_home_automation.txt
       Relevance: 0.417

Query: 'solar power tips'


  Found 3 results
    1. Reading solar data
- Daily generation is shaped like a hill: low at dawn, highest near solar noon, l...
       Source: tip_renewable_energy_integration.txt
       Relevance: 0.572
    2. Renewable Energy Integration: Getting the Most from Rooftop Solar

Solar panels produce the most ene...
       Source: tip_renewable_energy_integration.txt
       Relevance: 0.562
    3. Seasonal Energy Management

Energy use and solar output change with the seasons, so the right schedu...
       Source: tip_seasonal_energy_management.txt
       Relevance: 0.541
